In [1]:
import os, gc, time, torch, tqdm, numpy as np, pandas as pd
import torch.nn as nn
from torch_geometric.data import HeteroData
import torch.nn.functional as F
from neomodel import config as neomodel_config
from neomodel import db

# ==== 1. Neo4j 연결 ====
from app.config.local import NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD
neomodel_config.DATABASE_URL = f'bolt://{NEO4J_USER}:{DB_PASSWORD}@{NEO4J_HOST}:{NEO4J_PORT}'
try:
    result, _ = db.cypher_query("RETURN 1 AS test")
    print("✅ Neo4j 연결 성공!")
except Exception as e:
    print("❌ Neo4j 연결 실패:", e); exit()

# ==== 2. 헬퍼 ====
def cypher_df(q: str, params: dict | None = None) -> pd.DataFrame:
    records, cols = db.cypher_query(q, params or {})
    df = pd.DataFrame(records, columns=cols)
    id_cols = [c for c in df.columns if c.endswith('_id') or c in {'mid', 'oid', 'pid'}]
    for col in id_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64").astype("int64")
    return df

def safe_map(series: pd.Series, mapping: dict[int,int], col: str) -> np.ndarray:
    mapped = series.map(mapping)
    miss   = mapped.isna().sum()
    if miss:
        print(f"⚠️  {col}: {miss:,}개 매핑 실패 → 행 제거")
        mapped = mapped.dropna()
    return mapped.astype(np.int64).to_numpy()

# ==== 3. 전체 인덱스 & 피처 ====
print("📊 전체 멤버/주문/상품 추출...")
df_member  = cypher_df("MATCH (m:Member) RETURN m.member_id AS mid")
df_order   = cypher_df("MATCH (o:Order)  RETURN o.order_id AS oid")
df_product = cypher_df("MATCH (p:Product) RETURN p.product_id AS pid")
m2i = {mid: i for i, mid in enumerate(df_member.mid.unique())}
o2i = {oid: i for i, oid in enumerate(df_order.oid.unique())}
p2i = {pid: i for i, pid in enumerate(df_product.pid.unique())}
m2i_inv = {v: k for k, v in m2i.items()}
p2i_inv = {v: k for k, v in p2i.items()}
print(f"Total Members: {len(m2i):,} | Orders: {len(o2i):,} | Products: {len(p2i):,}")

print("📊 피처 로딩 중...")
df_order_feat = cypher_df("""
MATCH (o:Order)
RETURN o.order_id AS oid,
       o.days_since_prior_order_norm AS dsp,
       o.order_dow_norm AS dow,
       o.order_hour_of_day_norm AS hr,
       o.order_number_norm AS idx
""")
df_order_feat["oid_idx"] = safe_map(df_order_feat.oid, o2i, "oid")
order_feat = (
    df_order_feat.sort_values("oid_idx")[["dsp","dow","hr","idx"]]
    .to_numpy(np.float32)
)
df_prod_vec = cypher_df("""
MATCH (p:Product)
RETURN p.product_id AS pid,
       p.category_vector + p.name_vector AS vec
""")
df_prod_vec["pid_idx"] = safe_map(df_prod_vec.pid, p2i, "pid")
prod_feat = torch.tensor(
    np.stack(df_prod_vec.sort_values("pid_idx").vec.values),
    dtype=torch.float32
)
print(f"Order feature: {order_feat.shape}")
print(f"Product vector: {prod_feat.shape}")

print("📊 구조적 간선 생성...")
df_edge = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")
uo = torch.stack((
    torch.from_numpy(safe_map(df_edge.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge.oid, o2i, "oid"))
), 0).to(torch.long)
op = torch.stack((
    torch.from_numpy(safe_map(df_edge.oid, o2i, "oid")),
    torch.from_numpy(safe_map(df_edge.pid, p2i, "pid"))
), 0).to(torch.long)
print(f"uo: {uo.shape}, op: {op.shape}")



✅ config_module: ['CARD_SERVER_ADDRESS', 'DB_HOST', 'DB_NAME', 'DB_PASSWORD', 'DB_PORT', 'DB_USER', 'DEBUG', 'GEMINI_API_KEY', 'GOOGLE_APPLICATION_CREDENTIALS', 'GOOGLE_PROJECT', 'NEO4J_HOST', 'NEO4J_PORT', 'NEO4J_USER', 'SWAGGER_SPECS_JSON_ROUTE', 'SWAGGER_SPECS_ROUTE', 'SWAGGER_STATIC_PATH', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os']
✅ Neo4j 연결 성공!
📊 전체 멤버/주문/상품 추출...
Total Members: 206,209 | Orders: 3,346,083 | Products: 49,685
📊 피처 로딩 중...
Order feature: (3346083, 4)
Product vector: torch.Size([49685, 2048])
📊 구조적 간선 생성...
uo: torch.Size([2, 33819106]), op: torch.Size([2, 33819106])


In [2]:
# ==== 4. 모델 정의 및 로드 ====
class HetSAGE(nn.Module):
    def __init__(self, hid=128, out=128, p_drop=0.2, aggr='sum'):
        super().__init__()
        from torch_geometric.nn import HeteroConv, SAGEConv
        self.conv1 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
        }, aggr=aggr)
        self.conv2 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((hid, hid), out, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
        }, aggr=aggr)
        self.norm1 = nn.LayerNorm(hid)
        self.norm2 = nn.LayerNorm(out)
        self.p_drop = p_drop

    def _drop(self, h: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        return {k: F.dropout(v, p=self.p_drop, training=self.training) for k, v in h.items()}

    def forward(self, x_dict, edge_index_dict):
        h = self.conv1(x_dict, edge_index_dict)
        h = {k: F.relu(self.norm1(v)) for k, v in h.items()}
        h = self._drop(h)
        h = self.conv2(h, edge_index_dict)
        h = {k: F.relu(self.norm2(v)) for k, v in h.items()}
        h = self._drop(h)
        return h

ckpt_path = "checkpoints/best_model.pt"
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
hid_dim = ckpt.get('hid_dim', 128)
out_dim = ckpt.get('out_dim', 128)
model = HetSAGE(hid=hid_dim, out=out_dim)
model.load_state_dict(ckpt['model_state'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print("✔️  모델 로드 및 준비 완료.")

# ==== 5. 서브그래프 생성 ====
def build_subgraph_for_users(user_ids, uo, op, order_feat, prod_feat, m2i, o2i, p2i):
    user_idx = np.array([m2i[u] for u in user_ids])
    mask_user = np.isin(uo[0].cpu().numpy(), user_idx)
    uo_sub = uo[:, mask_user]
    order_idx = np.unique(uo_sub[1].cpu().numpy())
    mask_order = np.isin(op[0].cpu().numpy(), order_idx)
    op_sub = op[:, mask_order]
    product_idx = np.unique(op_sub[1].cpu().numpy())
    old2new_user   = {old: new for new, old in enumerate(user_idx)}
    old2new_order  = {old: new for new, old in enumerate(order_idx)}
    old2new_prod   = {old: new for new, old in enumerate(product_idx)}
    uo_sub_remap = torch.stack([
        torch.tensor([old2new_user[u.item()] for u in uo_sub[0]]),
        torch.tensor([old2new_order[o.item()] for o in uo_sub[1]])
    ], 0)
    op_sub_remap = torch.stack([
        torch.tensor([old2new_order[o.item()] for o in op_sub[0]]),
        torch.tensor([old2new_prod[p.item()] for p in op_sub[1]])
    ], 0)
    order_x   = torch.tensor(np.stack([order_feat[o] for o in order_idx]), dtype=torch.float32)
    product_x = torch.tensor(np.stack([prod_feat[p] for p in product_idx]), dtype=torch.float32)
    data = HeteroData()
    data['user'].num_nodes    = len(user_idx)
    data['order'].num_nodes   = len(order_idx)
    data['product'].num_nodes = len(product_idx)
    data['user'].x    = torch.nn.Embedding(len(user_idx), 32).weight
    data['order'].x   = order_x
    data['product'].x = product_x
    data['user','ordered','order'].edge_index         = uo_sub_remap
    data['order','ordered_by','user'].edge_index      = uo_sub_remap.flip(0)
    data['order','contains','product'].edge_index     = op_sub_remap
    data['product','contained_in','order'].edge_index = op_sub_remap.flip(0)
    return data, old2new_user, old2new_prod

✔️  모델 로드 및 준비 완료.


In [3]:
# ==== 6. 배치 예측 ====
@torch.no_grad()
def predict_for_all_members_subgraph(
    model, uo, op, order_feat, prod_feat,
    m2i, m2i_inv, p2i, p2i_inv,
    batch_size=200, top_k=5, device='cuda'
):
    user_ids = list(m2i.keys())
    n_users = len(user_ids)
    n_batches = int(np.ceil(n_users / batch_size))
    predictions = {}
    print(f"🔄 전체 유저 {n_users:,}명, {batch_size}명씩 서브그래프 배치 예측 시작...")
    for batch_idx in tqdm.tqdm(range(n_batches), desc="🔮 [예측 진행]"):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, n_users)
        batch_user_ids = user_ids[start:end]
        sub_data, user_remap, prod_remap = build_subgraph_for_users(
            batch_user_ids, uo, op, order_feat, prod_feat, m2i, o2i, p2i
        )
        sub_data = sub_data.to(device)
        model.eval()
        z = model(sub_data.x_dict, sub_data.edge_index_dict)
        uemb = z['user'].cpu()
        pemb = z['product'].cpu()
        del z; torch.cuda.empty_cache()
        for i, user_id in enumerate(batch_user_ids):
            user_vec = uemb[i]
            scores_all, idx_all = [], []
            CHUNK = 100_000
            for s in range(0, len(pemb), CHUNK):
                sc, ix = (pemb[s:s+CHUNK] @ user_vec).topk(top_k)
                scores_all.append(sc)
                idx_all.append(ix + s)
            scores_all = torch.cat(scores_all)
            idx_all = torch.cat(idx_all)
            topk_indices = scores_all.topk(top_k).indices
            top_global_pids = [list(prod_remap.keys())[idx_all[idx].item()] for idx in topk_indices]
            top_global_pids = [p2i_inv[pid] for pid in top_global_pids]
            predictions[user_id] = top_global_pids
    print("🎯 전체 예측 완료!")
    return predictions

predictions = predict_for_all_members_subgraph(
    model, uo, op, order_feat, prod_feat,
    m2i, m2i_inv, p2i, p2i_inv,
    batch_size=200, top_k=5, device=device
)

🔄 전체 유저 206,209명, 200명씩 서브그래프 배치 예측 시작...


🔮 [예측 진행]: 100%|███████████████████████| 1032/1032 [22:05<00:00,  1.28s/it]

🎯 전체 예측 완료!


In [4]:
# ==== 7. Neo4j 저장 ====
def save_predictions_to_neo4j(predictions, model_name="HetSAGE_v0.1"):
    print("📥 Neo4j에 예측결과 저장 중...")
    for member_id, product_ids in tqdm.tqdm(predictions.items(), desc="→ DB 저장"):
        q = """
        MATCH (m:Member {member_id: $member_id})-[:ORDERED]->(o:Order)
        WHERE NOT (o)-[:NEXT]->(:Order)
        WITH m, o ORDER BY o.order_number DESC LIMIT 1
        SET m.predict_order_list = $products,
            m.model = $model_name,
            o.predict_order_list = $products,
            o.model = $model_name
        """
        db.cypher_query(q, {
            "member_id": str(member_id),
            "products": [str(pid) for pid in product_ids],
            "model_name": model_name
        })
    print("✔️  Neo4j 저장 완료")



save_predictions_to_neo4j(predictions, model_name="HetSAGE_vFinal")
print("🧊 파이프라인 종료.")

📥 Neo4j에 예측결과 저장 중...


→ DB 저장: 100%|██████████████████████| 206209/206209 [8:42:56<00:00,  6.57it/s]

✔️  Neo4j 저장 완료
🧊 파이프라인 종료.


In [ ]:
def save_order_predictions(order_predictions, model_name="HetSAGE_vOrder"):
    for order_id, product_ids in tqdm.tqdm(order_predictions.items()):
        q = """
        MATCH (o:Order {order_id: $order_id})
        SET o.predict_order_list = $products,
            o.model = $model_name
        """
        db.cypher_query(q, {
            "order_id": str(order_id),
            "products": [str(pid) for pid in product_ids],
            "model_name": model_name
        })
